In [1]:
import os

import pandas as pd

In [2]:
from goatools.obo_parser import OBOReader, GODag

In [3]:
REF_DIR = "/n/groups/kirschner/reference/GO"

INPUT_DIR = "/n/groups/kirschner/Will/BedRest"

output_dir = os.path.join(INPUT_DIR, "WGCNA", "deseq2_50_cutHeight05")

# GO IDs

In [4]:
obodag = GODag(os.path.join(REF_DIR, "go.obo"), load_obsolete=True)

/n/groups/kirschner/reference/GO/go.obo: fmt(1.2) rel(2025-10-10) 51,842 Terms


## BP, MF

In [15]:
go_bp_id_to_name = []
i = 0

for go_id, term in obodag.items():
    # term = obodag[go_id]
    
    if term.namespace != "cellular_component":
        # print(term.namespace)
        continue
        
    # print(f"GO ID: {term.id}")
    # print(f"Name: {term.name}")
    # print(f"Definition: {term.definition}")
    # print(f"Parents: {[p.id for p in term.parents]}")
    go_bp_id_to_name.append({"id": term.id, "name": term.name, "obsolete": term.is_obsolete, "level": term.level})
    # if term.name == "regulation of immune response"
    # i += 1
    # break

    # if i == 10:
        # break

In [16]:
go_bp_id_to_name_df = pd.DataFrame(go_bp_id_to_name)
go_bp_id_to_name_df

,id,name,obsolete,level
0,GO:0000015,phosphopyruvate hydratase complex,False,3
1,GO:0000108,obsolete repairosome,True,0
2,GO:0000109,nucleotide-excision repair complex,False,3
3,GO:0000110,nucleotide-excision repair factor 1 complex,False,4
4,GO:0000111,nucleotide-excision repair factor 2 complex,False,4
...,...,...,...,...
4819,GO:1990316,Atg1/ULK1 kinase complex,False,7
4820,GO:1990429,peroxisomal importomer complex,False,3
4821,GO:1990477,MTREC complex,False,3
4822,GO:1990904,ribonucleoprotein complex,False,2


In [17]:
go_bp_id_to_name_df.level.value_counts().sort_index()

level
0      540
1        5
2      779
3     1053
4     1056
5      621
6      382
7      235
8       94
9       47
10      12
Name: count, dtype: int64

In [18]:
go_bp_id_to_name_df.loc[go_bp_id_to_name_df.level == 2, :]

,id,name,obsolete,level
14,GO:0000133,polarisome,False,2
47,GO:0000242,pericentriolar material,False,2
80,GO:0000399,cellular bud neck septin structure,False,2
81,GO:0000407,phagophore assembly site,False,2
83,GO:0000417,HIR complex,False,2
...,...,...,...,...
4765,GO:0042597,periplasmic space,False,2
4773,GO:0043220,Schmidt-Lanterman incisure,False,2
4807,GO:0098636,protein complex involved in cell adhesion,False,2
4822,GO:1990904,ribonucleoprotein complex,False,2


In [19]:
go_bp_id_to_name_df["msigdb_name"] = "GOCC_" + \
    go_bp_id_to_name_df["name"]\
        .str.replace("obsolete ", "")\
        .str.upper()\
        .str.replace("-", "_").str.replace(" ", "_")\
        .str.replace("(", "").str.replace(")", "")\
        .str.replace(",", "")\
        .str.replace("/", "_")

In [20]:
go_bp_id_to_name_df

,id,name,obsolete,level,msigdb_name
0,GO:0000015,phosphopyruvate hydratase complex,False,3,GOCC_PHOSPHOPYRUVATE_HYDRATASE_COMPLEX
1,GO:0000108,obsolete repairosome,True,0,GOCC_REPAIROSOME
2,GO:0000109,nucleotide-excision repair complex,False,3,GOCC_NUCLEOTIDE_EXCISION_REPAIR_COMPLEX
3,GO:0000110,nucleotide-excision repair factor 1 complex,False,4,GOCC_NUCLEOTIDE_EXCISION_REPAIR_FACTOR_1_COMPLEX
4,GO:0000111,nucleotide-excision repair factor 2 complex,False,4,GOCC_NUCLEOTIDE_EXCISION_REPAIR_FACTOR_2_COMPLEX
...,...,...,...,...,...
4819,GO:1990316,Atg1/ULK1 kinase complex,False,7,GOCC_ATG1_ULK1_KINASE_COMPLEX
4820,GO:1990429,peroxisomal importomer complex,False,3,GOCC_PEROXISOMAL_IMPORTOMER_COMPLEX
4821,GO:1990477,MTREC complex,False,3,GOCC_MTREC_COMPLEX
4822,GO:1990904,ribonucleoprotein complex,False,2,GOCC_RIBONUCLEOPROTEIN_COMPLEX


In [21]:
go_bp_id_to_name_df.to_csv(os.path.join(REF_DIR, "go_cc_id_to_name.csv"), index=False)

In [13]:
go_bp_id_to_name_df_ = go_bp_id_to_name_df.drop_duplicates("msigdb_name")

# GO BP results

In [44]:
def get_go_term_at_level(go_id, target_level, go_dag):
    """
    Finds the parent GO term at a specific level (depth) in the hierarchy.
    The top-level terms (Biological Process, Molecular Function, Cellular Component) 
    are at level 0 (or 1 depending on interpretation), so adjusting target_level might be necessary.
    Level here refers to the distance from the root of the specific ontology branch.
    """
    
    # Get all ancestors for the term (including itself)
    ancestors = go_dag[go_id].get_all_parents()
    # Add the term itself to the set
    ancestors.add(go_id)
    
    # Iterate through ancestors to find the one at the target level
    for term_id in ancestors:
        term = go_dag.get(term_id)
        if term is not None and term.level == target_level and (not term.is_obsolete):
            return term_id
            
    return None

## pre

In [24]:
pre_output_dir = os.path.join(output_dir, "pre", "ORA")

In [25]:
go_bp_ora_results_pre_df = pd.read_csv(os.path.join(pre_output_dir, "go_bp.csv"), index_col=0)

In [26]:
go_bp_ora_results_pre_unique = go_bp_ora_results_pre_df.sort_values("pval").drop_duplicates("gene_set")

In [27]:
print(go_bp_ora_results_pre_unique.shape[0])
go_bp_ora_results_pre_unique = go_bp_ora_results_pre_unique.loc[go_bp_ora_results_pre_unique.gene_set.isin(go_bp_id_to_name_df_.msigdb_name),:]
print(go_bp_ora_results_pre_unique.shape[0])

837
835


In [31]:
go_bp_ora_results_pre_unique["go_id"] = go_bp_id_to_name_df_.set_index("msigdb_name").loc[go_bp_ora_results_pre_unique.gene_set, "id"].values
go_bp_ora_results_pre_unique["level"] = go_bp_id_to_name_df_.set_index("msigdb_name").loc[go_bp_ora_results_pre_unique.gene_set, "level"].values

In [32]:
go_bp_ora_results_pre_unique.loc[go_bp_ora_results_pre_unique.go_id=="GO:0001501",:]

,statistic,pval,table,intersect,gene_set,pval_adj,module,moduleColor,go_id,level
3507,4.450566,6.539853e-13,39/324 vs 422/15603,39,GOBP_SKELETAL_SYSTEM_DEVELOPMENT,1.248458e-09,28,dimgrey,GO:0001501,4


In [35]:
go_bp_ora_results_pre_unique.value_counts("level").sort_index()

level
0       6
2      22
3     109
4     184
5     215
6     169
7      86
8      32
9       7
10      5
Name: count, dtype: int64

In [42]:
go_bp_ora_results_pre_unique.loc[go_bp_ora_results_pre_unique.level == 0,:]

,statistic,pval,table,intersect,gene_set,pval_adj,module,moduleColor,go_id,level
3639,11.778050,3.425377e-07,9/314 vs 39/16026,9,GOBP_THIOESTER_BIOSYNTHETIC_PROCESS,0.000163,22,darkred,GO:0035384,0
378,3.810453,9.033240e-06,17/346 vs 204/15821,17,GOBP_CELL_CELL_ADHESION_VIA_PLASMA_MEMBRANE_AD...,0.001724,28,dimgrey,GO:0098742,0
1385,5.926125,3.823050e-05,10/899 vs 29/15450,10,GOBP_NADH_METABOLIC_PROCESS,0.003164,68,orange,GO:0006734,0
1849,4.140887,3.055642e-04,10/313 vs 123/15942,10,GOBP_ORGANIC_ACID_TRANSMEMBRANE_TRANSPORT,0.032407,22,darkred,GO:1903825,0
429,7.212975,4.413600e-04,6/571 vs 23/15788,6,GOBP_CENTROMERE_COMPLEX_ASSEMBLY,0.027625,32,gainsboro,GO:0034508,0
137,5.081432,5.352010e-04,8/901 vs 27/15452,8,GOBP_ASPARTATE_FAMILY_AMINO_ACID_METABOLIC_PRO...,0.030050,68,orange,GO:0009066,0


In [66]:
target_level = 2
target_go_terms = []
id_col_name = f"go_term_level_{target_level}_id"
name_col_name = f"go_term_level_{target_level}_name"


for i, go_result in go_bp_ora_results_pre_unique.iterrows():
    
    if go_result.level <= target_level:
        target_go_terms.append(go_result.go_id)
    else:
        parent_term = get_go_term_at_level(go_result.go_id, target_level, obodag)
        target_go_terms.append(parent_term)

go_bp_ora_results_pre_unique[id_col_name] = target_go_terms
go_bp_ora_results_pre_unique[name_col_name] = go_bp_id_to_name_df_.set_index("id").loc[go_bp_ora_results_pre_unique[id_col_name], "name"].values

In [67]:
go_bp_ora_results_pre_unique.value_counts(name_col_name)

go_term_level_2_name
regulation of biological process                       326
metabolic process                                      136
cellular component organization or biogenesis           44
anatomical structure development                        38
leukocyte activation                                    25
                                                      ... 
production of molecular mediator of immune response      1
response to biotic stimulus                              1
response to abiotic stimulus                             1
system process                                           1
vesicle-mediated transport                               1
Name: count, Length: 67, dtype: int64

In [72]:
with pd.option_context('display.max_rows', None):
    display(go_bp_ora_results_pre_unique.value_counts(["go_term_level_2_name", "go_term_level_3_name"]).sort_index())

go_term_level_2_name                                                go_term_level_3_name                                                                       
T cell selection                                                    T cell selection                                                                                 1
                                                                    positive T cell selection                                                                        1
                                                                    thymic T cell selection                                                                          1
amyloid-beta clearance                                              amyloid-beta clearance                                                                           1
anatomical structure development                                    animal organ development                                                                         8
     

In [77]:
go_bp_ora_results_pre_unique.to_csv(os.path.join(pre_output_dir, "go_bp_parent_terms.csv"))

In [ ]:
# for i, row in go_bp_ora_results_pre_unique.iterrows():
#     print(row["go_id"], f'{row["pval"]:.10f}')

In [78]:
go_bp_ora_results_pre_unique

,statistic,pval,table,intersect,gene_set,pval_adj,module,moduleColor,go_id,level,go_term_level_2,go_term_level_3,go_term_level_3_id,go_term_level_3_name,go_term_level_2_id,go_term_level_2_name
519,36.288154,1.001112e-89,97/605 vs 69/15617,97,GOBP_CYTOPLASMIC_TRANSLATION,3.822246e-86,48,lightgrey,GO:0002181,6,GO:0008152,GO:0043170,GO:0043170,macromolecule metabolic process,GO:0008152,metabolic process
943,7.330181,2.866228e-39,89/816 vs 227/15256,89,GOBP_IMMUNE_RESPONSE_REGULATING_CELL_SURFACE_R...,1.094326e-35,5,black,GO:0002768,4,GO:0050789,GO:0002682,GO:0002682,regulation of immune system process,GO:0050789,regulation of biological process
30,5.972550,3.334568e-37,99/806 vs 312/15171,99,GOBP_ADAPTIVE_IMMUNE_RESPONSE,6.365690e-34,5,black,GO:0002250,3,GO:0006955,GO:0002250,GO:0002250,adaptive immune response,GO:0006955,immune response
3420,6.094299,9.490380e-35,89/613 vs 365/15321,89,GOBP_RIBONUCLEOPROTEIN_COMPLEX_BIOGENESIS,1.811714e-31,48,lightgrey,GO:0022613,4,GO:0071840,GO:0044085,GO:0044085,cellular component biogenesis,GO:0071840,cellular component organization or biogenesis
1066,5.689281,1.897512e-30,83/822 vs 270/15213,83,GOBP_LEUKOCYTE_MEDIATED_IMMUNITY,2.414901e-27,5,black,GO:0002443,3,GO:0002252,GO:0002443,GO:0002443,leukocyte mediated immunity,GO:0002252,immune effector process
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1605,5.728889,3.928824e-03,5/900 vs 15/15468,5,GOBP_NEGATIVE_REGULATION_OF_PHAGOCYTOSIS,4.717060e-02,5,black,GO:0050765,6,GO:0050789,GO:0048519,GO:0048519,negative regulation of biological process,GO:0050789,regulation of biological process
1362,5.728889,3.928824e-03,5/900 vs 15/15468,5,GOBP_MYD88_DEPENDENT_TOLL_LIKE_RECEPTOR_SIGNAL...,4.717060e-02,5,black,GO:0002755,7,GO:0050789,GO:0002757,GO:0002757,immune response-activating signaling pathway,GO:0050789,regulation of biological process
2914,5.728889,3.928824e-03,5/900 vs 15/15468,5,GOBP_REGULATION_OF_LYMPHOCYTE_CHEMOTAXIS,4.717060e-02,5,black,GO:1901623,6,GO:0050789,GO:0002682,GO:0002682,regulation of immune system process,GO:0050789,regulation of biological process
551,5.728889,3.928824e-03,5/900 vs 15/15468,5,GOBP_DETECTION_OF_EXTERNAL_BIOTIC_STIMULUS,4.717060e-02,5,black,GO:0098581,4,GO:0009605,GO:0043207,GO:0043207,response to external biotic stimulus,GO:0009605,response to external stimulus
